# Context Managers in Python
Covers: with statement, __enter__/__exit__, contextlib.contextmanager, suppress, ExitStack, nested context managers

## 1. __enter__ and __exit__ Protocol

In [ ]:
class ManagedResource:
    """Demonstrates the context manager protocol."""
    
    def __init__(self, name):
        self.name = name
        self.active = False
    
    def __enter__(self):
        print(f'[__enter__] Acquiring {self.name}')
        self.active = True
        return self  # bound to 'as' variable
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.active = False
        print(f'[__exit__] Releasing {self.name}')
        if exc_type is not None:
            print(f'  Exception: {exc_type.__name__}: {exc_val}')
        # Return True to suppress, False/None to propagate
        return False
    
    def use(self):
        if not self.active:
            raise RuntimeError('Resource not active')
        return f'Using {self.name}'

# Normal usage
print('=== Normal usage ===')
with ManagedResource('DB Connection') as res:
    print(res.use())
print('Active after with:', res.active)

# With exception
print('\n=== With exception ===')
try:
    with ManagedResource('File Handle') as res:
        print(res.use())
        raise ValueError('Something went wrong')
except ValueError:
    print('Exception propagated (not suppressed)')

## 2. Suppressing Exceptions in __exit__

In [ ]:
class SuppressErrors:
    """Context manager that suppresses specified exception types."""
    
    def __init__(self, *exception_types):
        self.exception_types = exception_types
        self.suppressed = None
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type and issubclass(exc_type, self.exception_types):
            self.suppressed = exc_val
            print(f'Suppressed: {exc_type.__name__}: {exc_val}')
            return True  # suppress!
        return False  # propagate other exceptions

# Suppress ValueError
with SuppressErrors(ValueError, KeyError) as ctx:
    raise ValueError('This is suppressed')
print('Execution continues. Suppressed:', ctx.suppressed)

# Does NOT suppress TypeError
try:
    with SuppressErrors(ValueError):
        raise TypeError('This is NOT suppressed')
except TypeError as e:
    print('TypeError propagated:', e)

## 3. @contextmanager — Generator-Based

In [ ]:
from contextlib import contextmanager
import time

# Simple timer
@contextmanager
def timer(label=''):
    """Time a block of code."""
    start = time.perf_counter()
    try:
        yield  # with block runs here
    finally:
        elapsed = time.perf_counter() - start
        print(f'{label}: {elapsed:.4f}s')

with timer('List comprehension'):
    result = [x**2 for x in range(100_000)]

# Context manager that yields a value
@contextmanager
def managed_connection(host, port):
    """Simulate a database connection."""
    print(f'Connecting to {host}:{port}')
    conn = {'host': host, 'port': port, 'open': True, 'queries': []}
    try:
        yield conn  # bound to 'as' variable
    except Exception as e:
        print(f'Error: {e} — rolling back')
        raise
    finally:
        conn['open'] = False
        print(f'Disconnected. Executed {len(conn["queries"])} queries')

with managed_connection('localhost', 5432) as conn:
    conn['queries'].append('SELECT * FROM users')
    conn['queries'].append('INSERT INTO logs VALUES (?)')
    print(f'Connected: {conn["host"]}:{conn["port"]}')

# Transaction pattern
@contextmanager
def transaction(db_name='mydb'):
    print(f'BEGIN TRANSACTION on {db_name}')
    try:
        yield
        print('COMMIT')
    except Exception:
        print('ROLLBACK')
        raise

print('\n--- Successful transaction ---')
with transaction():
    print('Executing queries...')

print('\n--- Failed transaction ---')
try:
    with transaction():
        print('Executing queries...')
        raise RuntimeError('Query failed!')
except RuntimeError:
    print('Transaction rolled back')

## 4. contextlib.suppress

In [ ]:
from contextlib import suppress
import os
import tempfile

# suppress — clean way to ignore specific exceptions
tmpdir = tempfile.mkdtemp()
nonexistent = os.path.join(tmpdir, 'nonexistent.txt')

# Without suppress
try:
    os.remove(nonexistent)
except FileNotFoundError:
    pass  # ignore

# With suppress — more readable
with suppress(FileNotFoundError):
    os.remove(nonexistent)
print('No error raised — FileNotFoundError suppressed')

# Multiple exception types
with suppress(FileNotFoundError, PermissionError, OSError):
    os.remove('/root/protected')  # PermissionError suppressed

# Use case: optional cleanup
temp_files = [os.path.join(tmpdir, f'temp_{i}.txt') for i in range(3)]
# Create some files
for f in temp_files[:2]:
    open(f, 'w').close()

print('\nCleaning up temp files:')
for f in temp_files:
    with suppress(FileNotFoundError):
        os.remove(f)
        print(f'  Removed: {os.path.basename(f)}')
print('Cleanup complete (missing files silently skipped)')

## 5. ExitStack — Dynamic Context Managers

In [ ]:
from contextlib import ExitStack
import tempfile
from pathlib import Path

tmpdir = Path(tempfile.mkdtemp())

# Open multiple files dynamically
filenames = [tmpdir / f'file{i}.txt' for i in range(4)]

with ExitStack() as stack:
    files = [stack.enter_context(open(f, 'w')) for f in filenames]
    for i, f in enumerate(files):
        f.write(f'Content of file {i}\n')
    print(f'Wrote to {len(files)} files')
# All files closed automatically

print('Files created:', [f.name for f in filenames if f.exists()])

# Callbacks — run in LIFO order
print('\nCallback order (LIFO):')
with ExitStack() as stack:
    stack.callback(print, '  Cleanup 1 (registered first, runs last)')
    stack.callback(print, '  Cleanup 2')
    stack.callback(print, '  Cleanup 3 (registered last, runs first)')
    print('  Inside with block')

# Conditional context managers
def process_data(use_lock=True, use_timer=True):
    import threading
    lock = threading.Lock()
    
    with ExitStack() as stack:
        if use_lock:
            stack.enter_context(lock)
            print('  Lock acquired')
        if use_timer:
            stack.enter_context(timer('process_data'))
        # do work
        result = sum(range(10000))
    return result

from contextlib import contextmanager
import time

@contextmanager
def timer(label):
    start = time.perf_counter()
    yield
    print(f'  {label}: {time.perf_counter()-start:.4f}s')

print('\nConditional context managers:')
process_data(use_lock=True, use_timer=True)

## 6. Practical Examples

In [ ]:
from contextlib import contextmanager
import os
import io
import sys
import tempfile
import shutil

# 1. Temporary directory
@contextmanager
def temp_directory():
    tmpdir = tempfile.mkdtemp()
    try:
        yield tmpdir
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

with temp_directory() as tmpdir:
    path = os.path.join(tmpdir, 'data.txt')
    with open(path, 'w') as f:
        f.write('temporary data')
    print('Temp file exists:', os.path.exists(path))
print('Temp dir cleaned up:', not os.path.exists(tmpdir))

# 2. Capture stdout
@contextmanager
def capture_stdout():
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        yield buffer
    finally:
        sys.stdout = old_stdout

with capture_stdout() as output:
    print('This goes to buffer')
    print('So does this')
print('Captured:', repr(output.getvalue()))

# 3. Environment variable override
@contextmanager
def env_override(**kwargs):
    old_values = {k: os.environ.get(k) for k in kwargs}
    os.environ.update({k: str(v) for k, v in kwargs.items()})
    try:
        yield
    finally:
        for key, old_val in old_values.items():
            if old_val is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = old_val

with env_override(MY_TEST_VAR='hello', DEBUG='true'):
    print('MY_TEST_VAR:', os.environ.get('MY_TEST_VAR'))
    print('DEBUG:', os.environ.get('DEBUG'))
print('After:', os.environ.get('MY_TEST_VAR', 'not set'))